In [ ]:
from tqdm import tqdm
import numpy as np
import torch
import torchvision
import warnings

warnings.filterwarnings('ignore')
NUM_CLASSES = 10

In [ ]:
model = torchvision.models.resnet18(pretrained=True)
print(model)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 177MB/s]


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

## Experiments:

1. Rerun training (restart kernel and run all cells) but this time, when loading the model in the first block of code, specify 'pretrained = True' in order to make use of the weights pretrained on Imagenet.
2. Rerun the code using the pretrained model but this time use a learning rate of 1e-3. What happens?
3. Rerun using the pretrained model and a lr of 1e-4 but this time only change the last layer in the model instead of the entire classifier.
3. Rerun the code using the pretrained model and a lr of 1e-4. This time, freeze the pretrained layers and only update the new layers for the first epochs. Afterwards, proceed to update the entire model. You can freeze parameters by specifying 'requires_grad = False'.
4. Rerun experiment 3 but gradually unfreeze layers instead of unfreezeing the entire model at once.

In [ ]:
def prepare_cifar10_dataset(root_dir='./dataset'):
    transform = torchvision.transforms.Compose([
        torchvision.transforms.PILToTensor(),
        torchvision.transforms.ConvertImageDtype(torch.float),
        torchvision.transforms.Resize((224, 224)),
        torchvision.transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
    ])

    train_dataset = torchvision.datasets.CIFAR10(
        root_dir,
        train=True,
        download=True,
        transform=transform
    )

    test_dataset = torchvision.datasets.CIFAR10(
        root_dir,
        train=False,
        download=True,
        transform=transform
    )

    n_train_samples = len(train_dataset)
    n_train_samples_split = int(.8 * n_train_samples)
    n_val_samples = n_train_samples - n_train_samples_split

    train_ds, val_ds = torch.utils.data.random_split(train_dataset, [
        n_train_samples_split, n_val_samples
    ])

    train_dl = torch.utils.data.DataLoader(train_ds, batch_size=32, shuffle=True)
    val_dl = torch.utils.data.DataLoader(val_ds, batch_size=32)
    test_dl = torch.utils.data.DataLoader(test_dataset, batch_size=32)

    return train_dl, val_dl, test_dl

In [ ]:
def iterate_dataloader(dataloader, model, criterion, optimizer=None, is_train=False, device=None):
    running_loss = 0.0
    correct = 0
    total = 0

    if is_train:
        model.train()
    else:
        model.eval()

    with torch.set_grad_enabled(is_train):
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    return running_loss, correct, total

In [ ]:
def train_model(model, train_dl, val_dl, optimizer, criterion, num_epochs=2):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    for epoch in range(num_epochs):
        running_loss, _, _ = iterate_dataloader(
            train_dl, model, criterion, optimizer=optimizer, is_train=True, device=device)
        epoch_loss = running_loss / len(train_dl.dataset)
        print(f"Epoch {epoch + 1}/{num_epochs}, Training Loss: {epoch_loss:.4f}")
        running_loss, correct, total = iterate_dataloader(
            val_dl, model, criterion, is_train=False, device=device)
        accuracy = 100 * correct / total
        print(f"Epoch {epoch + 1}/{num_epochs}, Validation Accuracy: {accuracy:.2f}%")

    print("Finished Training")
    return model

In [ ]:
def evaluate_model(model, test_dl, criterion):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    running_loss, correct, total = iterate_dataloader(
        test_dl, model, criterion, is_train=False, device=device)
    accuracy = 100 * correct / total
    print(f"Test Accuracy: {accuracy:.2f}%")
    return accuracy

In [ ]:
def load_resnet18_model(pretrained=False):
    model = torchvision.models.resnet18(pretrained=pretrained)
    return model

In [ ]:
model = load_resnet18_model(pretrained=True)

num_ftrs = model.fc.in_features
model.fc = torch.nn.Linear(num_ftrs, NUM_CLASSES)

train_dl, val_dl, test_dl = prepare_cifar10_dataset()

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion = torch.nn.CrossEntropyLoss()

trained_model = train_model(model, train_dl, val_dl, optimizer, criterion, num_epochs=2)

evaluate_model(trained_model, test_dl, criterion)

100%|██████████| 170M/170M [00:03<00:00, 43.4MB/s]


Epoch 1/2, Training Loss: 0.3569
Epoch 1/2, Validation Accuracy: 93.17%
Epoch 2/2, Training Loss: 0.1301
Epoch 2/2, Validation Accuracy: 93.88%
Finished Training
Test Accuracy: 93.50%


93.5

In [ ]:
def train_gradual_unfreeze(model, train_dl, val_dl, optimizer, criterion, num_epochs=5, unfreeze_schedule=None):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    for epoch in range(num_epochs):
        if epoch in unfreeze_schedule:
            layer_name = unfreeze_schedule[epoch]
            if layer_name == 'fc':
                for param in model.fc.parameters():
                    param.requires_grad = True
                print(f"Epoch {epoch+1}: Unfroze fc layer")
            elif hasattr(model, layer_name):
                 for param in getattr(model, layer_name).parameters():
                    param.requires_grad = True
                 print(f"Epoch {epoch+1}: Unfroze {layer_name}")

        running_loss, _, _ = iterate_dataloader(
            train_dl, model, criterion, optimizer=optimizer, is_train=True, device=device)
        epoch_loss = running_loss / len(train_dl.dataset)
        print(f"Epoch {epoch + 1}/{num_epochs}, Training Loss: {epoch_loss:.4f}")

        running_loss, correct, total = iterate_dataloader(
            val_dl, model, criterion, is_train=False, device=device)
        accuracy = 100 * correct / total
        print(f"Epoch {epoch + 1}/{num_epochs}, Validation Accuracy: {accuracy:.2f}%")

    print("Finished Training")
    return model

model = load_resnet18_model(pretrained=True)

for param in model.parameters():
    param.requires_grad = False

num_ftrs = model.fc.in_features
model.fc = torch.nn.Linear(num_ftrs, NUM_CLASSES)
for param in model.fc.parameters():
    param.requires_grad = True

train_dl, val_dl, test_dl = prepare_cifar10_dataset()

optimizer = torch.optim.Adam([
    {'params': model.fc.parameters(), 'lr': 1e-4},
    {'params': model.layer4.parameters(), 'lr': 1e-5},
    {'params': model.layer3.parameters(), 'lr': 1e-6},
    {'params': model.layer2.parameters(), 'lr': 1e-7},
    {'params': model.layer1.parameters(), 'lr': 1e-8},
], lr=1e-9)

criterion = torch.nn.CrossEntropyLoss()

unfreeze_schedule = {
    0: 'fc',
    1: 'layer4',
    2: 'layer3',
    3: 'layer2',
    4: 'layer1'
}

trained_model_gradual = train_gradual_unfreeze(
    model,
    train_dl,
    val_dl,
    optimizer,
    criterion,
    num_epochs=5,
    unfreeze_schedule=unfreeze_schedule)

evaluate_model(trained_model_gradual, test_dl, criterion)

Epoch 1: Unfroze fc layer
Epoch 1/5, Training Loss: 1.4654
Epoch 1/5, Validation Accuracy: 73.71%
Epoch 2: Unfroze layer4
Epoch 2/5, Training Loss: 0.5843
Epoch 2/5, Validation Accuracy: 87.38%
Epoch 3: Unfroze layer3
Epoch 3/5, Training Loss: 0.3617
Epoch 3/5, Validation Accuracy: 89.29%
Epoch 4: Unfroze layer2
Epoch 4/5, Training Loss: 0.2661
Epoch 4/5, Validation Accuracy: 90.13%
Epoch 5: Unfroze layer1
Epoch 5/5, Training Loss: 0.2009
Epoch 5/5, Validation Accuracy: 90.97%
Finished Training
Test Accuracy: 90.12%


90.12